# Experimento N6.1 — Validación independiente de retención de experiencias de equilibrio en DDPG-R2

**Pregunta confirmatoria:** ¿la reserva FIFO de experiencias locales y su uso en un 25 % del lote mejora consistentemente DDPG con Curriculum Learning en **cinco semillas nuevas**?

Este notebook reproduce la física, recompensa R2, representación física, currículo de cuatro etapas y configuración H02 de **N6**, sin buscar nuevos hiperparámetros. Las políticas de N6 **no se reutilizan**: todos los modelos de N6.1 empiezan desde cero.

**Diseño:** 5 semillas de entrenamiento (1101–1105) × 2 tratamientos (N5_CURRICULO y N6_RETENCION) × 160 000 interacciones = **1 600 000 pasos**. Checkpoints 40/80/120/160K; cinco episodios de validación 8200–8204 por checkpoint y diez episodios finales reservados 9980–9989 por política seleccionada.

**Importante:** el desempeño final corresponde al checkpoint seleccionado **solo con validación**, no necesariamente al modelo de 160K. Informaremos por separado la pérdida de rendimiento entre el mejor checkpoint y 160K. Las semillas de evaluación no son réplicas independientes de entrenamiento.

**Seguridad:** conexión obligatoria a Google Drive, checkpoints verificados, ramas completadas reutilizables, posibilidad de ejecutar evaluación en una sesión posterior. Si se interrumpe una rama durante el entrenamiento, se reinicia esa rama desde cero (no se restauran optimizador ni replay buffer).

**Criterios de confirmación previamente fijados:** mejora media pareada ≥10 s, mejora estricta en ≥4 de 5 semillas, y sin disminuir el número de políticas con estabilidad media ≥100 s. El cumplimiento será una señal de confirmación **limitada a cinco semillas**, no prueba de superioridad universal.


## 1. Instalar dependencias en Google Colab


In [ ]:
%pip -q install 'stable-baselines3>=2.3,<3' 'gymnasium>=0.29,<2' 'pandas>=2' 'matplotlib>=3'

## 2. Configuración y guardado persistente

Autoriza el acceso a Google Drive. Verifica que la carpeta activa sea `MyDrive/N61_VALIDACION_RETENCION_DDPG_R2`, que las semillas sean 1101–1105 y que las semillas reservadas para la evaluación final sean 9980–9989. No ejecutes entrenamiento hasta superar las pruebas de las secciones 4 y 5.


In [ ]:
from pathlib import Path
import os, json, math, hashlib, time, random, zipfile, platform, shutil, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
import stable_baselines3 as sb3
from stable_baselines3 import DDPG
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.buffers import ReplayBuffer
from stable_baselines3.common.type_aliases import ReplayBufferSamples
import torch
from google.colab import files, drive

drive.mount('/content/drive')
OUT=Path('/content/drive/MyDrive/N61_VALIDACION_RETENCION_DDPG_R2')
OUT.mkdir(parents=True,exist_ok=True)
assert OUT.is_dir() and str(OUT.resolve()).startswith('/content/drive/MyDrive/')

CFG=dict(train_seeds=[1101,1102,1103,1104,1105], total_steps=160000,
    checkpoints=[40000,80000,120000,160000], learning_starts=1000,
    val_seeds=list(range(8200,8205)), test_seeds=list(range(9980,9990)),
    episode_steps=6000,dt=.02,
    theta_bono_rad=.15, omega_bono_r1_rad_s=.50,
    x_bono_r2_m=.35,v_bono_r2_m_s=.25,bono_r2=3.,
    x_estricto=.20,v_estricto=.10,theta_estricto=.05,omega_estricto=.10)
H02=dict(learning_rate=3e-4,gamma=.995,tau=.010,batch_size=128,sigma=.10,net_arch=[128,128])
TRATAMIENTOS=('N5_CURRICULO','N6_RETENCION')
RETENCION=dict(capacity=20000,fraction=0.25,activate_at=40000)
ETAPAS={'equilibrio':(0,40000),'recuperacion':(40000,80000),
        'mezcla_swingup':(80000,120000),'tarea_original':(120000,160000)}
assert set(CFG['train_seeds']).isdisjoint(CFG['val_seeds']+CFG['test_seeds'])
assert set(CFG['val_seeds']).isdisjoint(CFG['test_seeds'])
print('Carpeta persistente:',OUT)
print('SB3',sb3.__version__,'Gymnasium',gym.__version__)
print('10 entrenamientos × 160K =',len(TRATAMIENTOS)*len(CFG['train_seeds'])*CFG['total_steps'],'pasos')


## 3. Entorno físico y recompensa R2 originales

Se reutiliza el código de N6, sin modificación de ecuaciones, dinámica, penalizaciones o bonificación R2. Se calcula su SHA-256 para identificar el entorno exacto utilizado. Ambas variantes se evaluarán desde abajo, con las condiciones iniciales originales.


In [ ]:
from pathlib import Path
Path('envs').mkdir(exist_ok=True)
Path('envs/__init__.py').write_text('',encoding='utf-8')
ORIGINAL_ENV = '"""\nEntorno Gymnasium: péndulo invertido tipo carro–poste con acción continua.\n\nEstado:\n    s_t = [x_t, x_dot_t, theta_t, theta_dot_t]\n\nAcción:\n    a_t = u(t), fuerza continua aplicada al carro.\n\nConvención angular:\n    theta = 0  -> péndulo vertical hacia arriba.\n    theta = pi -> péndulo hacia abajo.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Optional\n\nimport gymnasium as gym\nfrom gymnasium import spaces\nimport numpy as np\n\n\nclass CartPoleContinuousEnv(gym.Env):\n    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 50}\n\n    def __init__(\n        self,\n        render_mode: Optional[str] = None,\n        max_episode_steps: int = 1000,\n        force_max: float = 10.0,\n        dt: float = 0.02,\n        x_threshold: float = 2.4,\n        noise_std: float = 0.0,\n        viscous_friction_cart: float = 0.0,\n        viscous_friction_pole: float = 0.0,\n        external_impulse: float = 0.0,\n        impulse_step: Optional[int] = None,\n        random_initial_state: bool = True,\n        start_near_downward: bool = True,\n    ):\n        super().__init__()\n\n        self.gravity = 9.81\n        self.masscart = 1.0\n        self.masspole = 0.1\n        self.total_mass = self.masscart + self.masspole\n        self.length = 0.5\n        self.polemass_length = self.masspole * self.length\n\n        self.force_max = float(force_max)\n        self.dt = float(dt)\n        self.x_threshold = float(x_threshold)\n        self.max_episode_steps = int(max_episode_steps)\n\n        self.noise_std = float(noise_std)\n        self.viscous_friction_cart = float(viscous_friction_cart)\n        self.viscous_friction_pole = float(viscous_friction_pole)\n        self.external_impulse = float(external_impulse)\n        self.impulse_step = impulse_step\n\n        self.random_initial_state = random_initial_state\n        self.start_near_downward = start_near_downward\n\n        self.render_mode = render_mode\n        self.screen = None\n        self.clock = None\n        self.screen_width = 800\n        self.screen_height = 500\n\n        self.action_space = spaces.Box(\n            low=np.array([-self.force_max], dtype=np.float32),\n            high=np.array([self.force_max], dtype=np.float32),\n            dtype=np.float32,\n        )\n\n        high = np.array(\n            [\n                self.x_threshold * 2.0,\n                np.finfo(np.float32).max,\n                math.pi,\n                np.finfo(np.float32).max,\n            ],\n            dtype=np.float32,\n        )\n        self.observation_space = spaces.Box(-high, high, dtype=np.float32)\n\n        self.state: Optional[np.ndarray] = None\n        self.steps = 0\n        self.last_u = 0.0\n        self.cumulative_control_effort = 0.0\n        self.cumulative_reward = 0.0\n\n    @staticmethod\n    def angle_normalize(theta: float) -> float:\n        return ((theta + math.pi) % (2.0 * math.pi)) - math.pi\n\n    def _get_obs(self) -> np.ndarray:\n        assert self.state is not None\n        obs = self.state.copy()\n\n        if self.noise_std > 0:\n            noise = self.np_random.normal(0.0, self.noise_std, size=obs.shape)\n            obs = obs + noise.astype(np.float32)\n\n        obs[2] = self.angle_normalize(float(obs[2]))\n        return obs.astype(np.float32)\n\n    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):\n        super().reset(seed=seed)\n        self.steps = 0\n        self.last_u = 0.0\n        self.cumulative_control_effort = 0.0\n        self.cumulative_reward = 0.0\n\n        if self.random_initial_state:\n            if self.start_near_downward:\n                x = self.np_random.uniform(low=-0.05, high=0.05)\n                x_dot = self.np_random.uniform(low=-0.05, high=0.05)\n                theta = math.pi + self.np_random.uniform(low=-0.10, high=0.10)\n                theta_dot = self.np_random.uniform(low=-0.05, high=0.05)\n            else:\n                x = self.np_random.uniform(low=-0.5, high=0.5)\n                x_dot = self.np_random.uniform(low=-0.5, high=0.5)\n                theta = self.np_random.uniform(low=-math.pi, high=math.pi)\n                theta_dot = self.np_random.uniform(low=-1.0, high=1.0)\n        else:\n            x, x_dot, theta, theta_dot = 0.0, 0.0, math.pi, 0.0\n\n        theta = self.angle_normalize(theta)\n        self.state = np.array([x, x_dot, theta, theta_dot], dtype=np.float32)\n\n        info = {\n            "cumulative_control_effort": self.cumulative_control_effort,\n            "cumulative_reward": self.cumulative_reward,\n        }\n        return self._get_obs(), info\n\n    def step(self, action):\n        assert self.state is not None\n\n        x, x_dot, theta, theta_dot = map(float, self.state)\n        force = float(np.clip(action[0], -self.force_max, self.force_max))\n\n        if self.impulse_step is not None and self.steps == self.impulse_step:\n            force += self.external_impulse\n\n        force -= self.viscous_friction_cart * x_dot\n\n        costheta = math.cos(theta)\n        sintheta = math.sin(theta)\n\n        temp = (\n            force + self.polemass_length * theta_dot**2 * sintheta\n        ) / self.total_mass\n\n        theta_acc = (\n            self.gravity * sintheta\n            - costheta * temp\n            - self.viscous_friction_pole * theta_dot / max(self.polemass_length, 1e-8)\n        ) / (\n            self.length * (4.0 / 3.0 - self.masspole * costheta**2 / self.total_mass)\n        )\n\n        x_acc = temp - self.polemass_length * theta_acc * costheta / self.total_mass\n\n        x_dot = x_dot + self.dt * x_acc\n        x = x + self.dt * x_dot\n        theta_dot = theta_dot + self.dt * theta_acc\n        theta = theta + self.dt * theta_dot\n        theta = self.angle_normalize(theta)\n\n        self.state = np.array([x, x_dot, theta, theta_dot], dtype=np.float32)\n        self.steps += 1\n\n        reward = self._reward(x, x_dot, theta, theta_dot, force)\n\n        terminated = bool(abs(x) > self.x_threshold)\n        truncated = bool(self.steps >= self.max_episode_steps)\n\n        self.last_u = force\n        self.cumulative_control_effort += force**2 * self.dt\n        self.cumulative_reward += reward\n\n        info = {\n            "x": x,\n            "x_dot": x_dot,\n            "theta": theta,\n            "theta_dot": theta_dot,\n            "u": force,\n            "cumulative_control_effort": self.cumulative_control_effort,\n            "cumulative_reward": self.cumulative_reward,\n            "is_upright": abs(theta) < 0.15,\n        }\n\n        if self.render_mode == "human":\n            self.render()\n\n        return self._get_obs(), float(reward), terminated, truncated, info\n\n    def _reward(self, x: float, x_dot: float, theta: float, theta_dot: float, u: float) -> float:\n        """\n        Recompensa mejorada para swing-up y estabilización.\n        """\n\n        reward = 1.0 + math.cos(theta)\n\n        reward += 0.05\n\n        reward -= 0.20 * x**2\n        reward -= 0.01 * x_dot**2\n        reward -= 0.02 * theta_dot**2\n        reward -= 0.001 * u**2\n\n        if abs(theta) < 0.15:\n            reward += 3.0\n\n        if abs(x) > 0.80 * self.x_threshold:\n            reward -= 5.0 * (abs(x) - 0.80 * self.x_threshold)\n\n        if abs(x) > self.x_threshold:\n            reward -= 100.0\n\n        return reward\n\n    def render(self):\n        try:\n            import pygame\n            from pygame import gfxdraw\n        except ImportError as exc:\n            raise ImportError("Para renderizar instala pygame: pip install pygame") from exc\n\n        if self.screen is None:\n            pygame.init()\n            if self.render_mode == "human":\n                self.screen = pygame.display.set_mode((self.screen_width, self.screen_height))\n            else:\n                self.screen = pygame.Surface((self.screen_width, self.screen_height))\n            self.clock = pygame.time.Clock()\n\n        assert self.state is not None\n        x, _, theta, _ = self.state\n\n        world_width = self.x_threshold * 2\n        scale = self.screen_width / world_width\n        cart_y = int(self.screen_height * 0.65)\n        cart_w = 60\n        cart_h = 30\n        pole_len = scale * (2 * self.length)\n\n        cart_x = int(self.screen_width / 2 + x * scale)\n\n        self.screen.fill((255, 255, 255))\n        import pygame\n\n        pygame.draw.line(\n            self.screen, (0, 0, 0),\n            (0, cart_y + cart_h // 2),\n            (self.screen_width, cart_y + cart_h // 2),\n            2\n        )\n\n        cart_rect = pygame.Rect(cart_x - cart_w // 2, cart_y - cart_h // 2, cart_w, cart_h)\n        pygame.draw.rect(self.screen, (30, 144, 255), cart_rect)\n        pygame.draw.rect(self.screen, (0, 0, 0), cart_rect, 2)\n\n        pivot = (cart_x, cart_y - cart_h // 2)\n        pole_x = pivot[0] + pole_len * math.sin(theta)\n        pole_y = pivot[1] - pole_len * math.cos(theta)\n\n        pygame.draw.line(self.screen, (220, 80, 20), pivot, (pole_x, pole_y), 6)\n        gfxdraw.filled_circle(self.screen, int(pole_x), int(pole_y), 10, (220, 80, 20))\n        gfxdraw.aacircle(self.screen, int(pole_x), int(pole_y), 10, (0, 0, 0))\n\n        font = pygame.font.SysFont("Arial", 18)\n        txt = font.render(f"theta={theta:.2f} rad | u={self.last_u:.2f} N", True, (0, 0, 0))\n        self.screen.blit(txt, (20, 20))\n\n        if self.render_mode == "human":\n            pygame.event.pump()\n            pygame.display.flip()\n            self.clock.tick(self.metadata["render_fps"])\n        else:\n            return np.transpose(\n                np.array(pygame.surfarray.pixels3d(self.screen)), axes=(1, 0, 2)\n            )\n\n    def close(self):\n        if self.screen is not None:\n            import pygame\n            pygame.display.quit()\n            pygame.quit()\n            self.screen = None\n            self.clock = None\n'
Path('envs/cartpole_continuous_env.py').write_text(ORIGINAL_ENV,encoding='utf-8')
print('Entorno original preparado; SHA-256:',hashlib.sha256(ORIGINAL_ENV.encode('utf-8')).hexdigest())
from envs.cartpole_continuous_env import CartPoleContinuousEnv

## 4. Currículo de cuatro etapas y pruebas de equivalencia

Los dos tratamientos emplean **el mismo currículo**. Los cambios de etapa se solicitan a los pasos 40K, 80K y 120K y se aplican en el siguiente `reset()`. La evaluación final nunca utiliza resets facilitados.

La única diferencia es la forma de extraer parte de los minibatches de la memoria del agente después de 40K pasos. La prueba siguiente comprueba equivalencia física y resets de etapa sin entrenamiento largo.


In [ ]:
class CartPoleRewardR1(CartPoleContinuousEnv):
    def _reward(self,x,x_dot,theta,theta_dot,u):
        r=super()._reward(x,x_dot,theta,theta_dot,u)
        if abs(theta)<CFG['theta_bono_rad'] and abs(theta_dot)>=CFG['omega_bono_r1_rad_s']:
            r-=3.0
        return r

class CartPoleRewardR2(CartPoleRewardR1):
    def _reward(self,x,x_dot,theta,theta_dot,u):
        r=super()._reward(x,x_dot,theta,theta_dot,u)
        local=(abs(x)<CFG['x_bono_r2_m'] and abs(x_dot)<CFG['v_bono_r2_m_s']
               and abs(theta)<CFG['theta_bono_rad'] and abs(theta_dot)<CFG['omega_bono_r1_rad_s'])
        return r+(CFG['bono_r2'] if local else 0.0)

class CurriculumReset(gym.Wrapper):
    """Modifica SOLO los estados de reset. Solicitud de nueva etapa se activa en el siguiente episodio."""
    def __init__(self, env, enabled=False):
        super().__init__(env)
        self.enabled=bool(enabled)
        self.requested_stage='equilibrio' if enabled else 'tarea_original'
        self.active_stage=self.requested_stage
        self.transitions=[]
        self.global_training_step=0

    def set_stage(self, stage, step=None):
        assert stage in ETAPAS
        if not self.enabled:
            assert stage=='tarea_original'
        self.requested_stage=stage
        if step is not None: self.global_training_step=int(step)

    def reset(self, *, seed=None, options=None):
        obs,info=self.env.reset(seed=seed,options=options)
        original_stage=self.active_stage
        self.active_stage=self.requested_stage
        if original_stage!=self.active_stage:
            self.transitions.append(dict(requested_at_or_before=self.global_training_step,
                                         activated_stage=self.active_stage))
        if self.enabled and self.active_stage!='tarea_original':
            rng=self.unwrapped.np_random
            stage=self.active_stage
            if stage=='equilibrio':
                x=rng.uniform(-.05,.05);v=rng.uniform(-.03,.03)
                theta=rng.uniform(-.05,.05);omega=rng.uniform(-.05,.05)
            elif stage=='recuperacion':
                x=rng.uniform(-.35,.35);v=rng.uniform(-.25,.25)
                theta=rng.uniform(-.35,.35);omega=rng.uniform(-.60,.60)
            else:
                # 60 % abajo: swing-up, 40 % orientaciones variadas.
                if rng.random()<.6:
                    x=rng.uniform(-.10,.10);v=rng.uniform(-.10,.10)
                    theta=math.pi+rng.uniform(-.20,.20);omega=rng.uniform(-.20,.20)
                else:
                    x=rng.uniform(-.35,.35);v=rng.uniform(-.30,.30)
                    theta=rng.uniform(-math.pi,math.pi);omega=rng.uniform(-1.0,1.0)
            theta=self.unwrapped.angle_normalize(float(theta))
            self.unwrapped.state=np.array([x,v,theta,omega],dtype=np.float32)
            obs=self.unwrapped._get_obs()
        info=dict(info);info['curriculum_stage']=self.active_stage
        return obs,info

    def step(self,action):
        obs,r,term,trunc,info=self.env.step(action)
        info=dict(info);info['curriculum_stage']=self.active_stage
        return obs,r,term,trunc,info

class StateAudit(gym.Wrapper):
    def step(self,action):
        obs,reward,term,trunc,info=self.env.step(action)
        info=dict(info)
        x,v,th,w=map(float,self.unwrapped.state)
        info['local_r2_activado']=int(abs(x)<CFG['x_bono_r2_m'] and abs(v)<CFG['v_bono_r2_m_s']
                                       and abs(th)<CFG['theta_bono_rad'] and abs(w)<CFG['omega_bono_r1_rad_s'])
        info['cerca_vertical']=int(abs(th)<CFG['theta_bono_rad'])
        info['estricta']=int(abs(x)<=CFG['x_estricto'] and abs(v)<=CFG['v_estricto']
                            and abs(th)<=CFG['theta_estricto'] and abs(w)<=CFG['omega_estricto'])
        info['velocidad_carro_estricta']=int(abs(v)<=CFG['v_estricto'])
        return obs,reward,term,trunc,info

def solicitar_etapa(env, etapa, paso=None):
    """Encuentra CurriculumReset bajo StateAudit/Monitor sin depender de Gymnasium forwarding."""
    actual = env
    while actual is not None:
        if isinstance(actual, CurriculumReset):
            actual.set_stage(etapa, paso)
            return actual
        actual = getattr(actual, 'env', None)
    raise RuntimeError('No se encontró CurriculumReset en la cadena de wrappers')

def make_physical_env(steps=6000):
    return CartPoleRewardR2(max_episode_steps=steps,force_max=10.0,dt=0.02,
        x_threshold=2.4,noise_std=0.0,viscous_friction_cart=0.0,
        viscous_friction_pole=0.0,random_initial_state=True,start_near_downward=True)

def make_agent_env(tratamiento,steps=6000,train_audit=False):
    assert tratamiento in TRATAMIENTOS
    env=CurriculumReset(make_physical_env(steps),enabled=(train_audit and tratamiento in ('N5_CURRICULO','N6_RETENCION')))
    return StateAudit(env) if train_audit else env

# Igualdad de acciones, observaciones, física, R2 y reinicio habitual en evaluación.
a=make_agent_env('N5_CURRICULO');b=make_agent_env('N6_RETENCION')
for seed in [1101,8200,9980]:
    oa,_=a.reset(seed=seed);ob,_=b.reset(seed=seed)
    assert oa.shape==ob.shape==(4,) and np.array_equal(oa,ob)
    for u in [1.,-2.,3.,0.]:
        oa,ra,ta,ua,_=a.step(np.array([u],dtype=np.float32))
        ob,rb,tb,ub,_=b.step(np.array([u],dtype=np.float32))
        assert np.array_equal(a.unwrapped.state,b.unwrapped.state)
        assert np.array_equal(oa,ob) and np.isclose(ra,rb) and (ta,ua)==(tb,ub)
a.close();b.close()
# Prueba del currículo sin entrenamiento.
e=make_agent_env('N5_CURRICULO',train_audit=True)
for stage in ETAPAS:
    solicitar_etapa(e,stage)
    obs,info=e.reset(seed=901)
    assert obs.shape==(4,) and np.isfinite(obs).all() and info['curriculum_stage']==stage
    if stage=='equilibrio': assert abs(obs[2])<=.05 and abs(obs[0])<=.05
    if stage=='recuperacion': assert abs(obs[2])<=.35 and abs(obs[0])<=.35
    _,reward,_,_,inf=e.step(np.array([0.],dtype=np.float32))
    assert np.isfinite(reward) and inf['curriculum_stage']==stage
e.close()
c=make_agent_env('N5_CURRICULO',train_audit=True)
d=make_agent_env('N6_RETENCION',train_audit=True)
for etapa in ETAPAS:
    solicitar_etapa(c,etapa);solicitar_etapa(d,etapa)
    for sem in [1101,1102]:
        oc,ic=c.reset(seed=sem);od,id_=d.reset(seed=sem)
        assert np.array_equal(oc,od) and ic['curriculum_stage']==id_['curriculum_stage']==etapa
c.close();d.close()
print('PRUEBAS OK: misma fisica, recompensa, condiciones originales de evaluacion y resets por etapa.')


## 5. Memoria de retención y pruebas de integridad (SIN entrenamiento largo)

Se conservan transiciones cuyo estado actual y siguiente estado pertenecen a la **región local R2**, que es más amplia que el equilibrio estricto. La reserva FIFO tiene capacidad de 20 000. A partir de 40K pasos, mientras existan transiciones elegibles, el muestreo utiliza **75 % del replay uniforme y 25 % de la reserva**. No modifica el tamaño del minibatch (128) ni el número de pasos entrenados.

**Atención:** este experimento evalúa *adquisición* y *retención* por separado. Que el método permita seleccionar un checkpoint exitoso no demuestra que conserve la habilidad a 160K.


In [ ]:
class RetentionReplayBuffer(ReplayBuffer):
    """Replay de DDPG con copia FIFO de transiciones en region local R2.

    Muestreo 75/25 despues de 40K. No cambia arquitectura, recompensa ni batch total.
    Solo soporta n_envs=1 y observaciones fisicas de cuatro componentes.
    """
    def __init__(self, buffer_size, observation_space, action_space, device="auto",
                 n_envs=1, optimize_memory_usage=False, handle_timeout_termination=True,
                 stable_capacity=20000, fraction=0.25, **kwargs):
        super().__init__(buffer_size,observation_space,action_space,device=device,
                         n_envs=n_envs,optimize_memory_usage=optimize_memory_usage,
                         handle_timeout_termination=handle_timeout_termination)
        assert n_envs==1 and observation_space.shape==(4,)
        assert 0.0 < fraction < 1.0
        self.fraction=float(fraction)
        self.retention_active=False
        self.stable=ReplayBuffer(stable_capacity,observation_space,action_space,
            device=device,n_envs=n_envs,optimize_memory_usage=False,
            handle_timeout_termination=handle_timeout_termination)
        self.total_added=0
        self.local_added=0
        self.strict_added=0
        self.retained_sampled=0
        self.normal_sampled=0
        self.mixed_batches=0

    @staticmethod
    def in_region(s, strict=False):
        s=np.asarray(s).reshape(-1)
        if s.shape[0]!=4 or not np.isfinite(s).all():return False
        x,v,theta,omega=map(float,s)
        if strict:
            return abs(x)<=.20 and abs(v)<=.10 and abs(theta)<=.05 and abs(omega)<=.10
        return abs(x)<.35 and abs(v)<.25 and abs(theta)<.15 and abs(omega)<.50

    def add(self,obs,next_obs,action,reward,done,infos):
        super().add(obs,next_obs,action,reward,done,infos)
        self.total_added+=1
        # Sólo transiciones locales, sin transición que cierra el episodio.
        eligible=(not bool(np.asarray(done).reshape(-1)[0]) and
                  self.in_region(obs) and self.in_region(next_obs))
        if eligible:
            self.stable.add(obs,next_obs,action,reward,done,infos)
            self.local_added+=1
            if self.in_region(obs,True) and self.in_region(next_obs,True):
                self.strict_added+=1

    def sample(self,batch_size,env=None):
        use_stable=(self.retention_active and self.stable.size()>0)
        k=min(int(round(batch_size*self.fraction)),batch_size-1) if use_stable else 0
        base=super().sample(batch_size-k,env=env)
        if k==0:
            self.normal_sampled+=batch_size
            return base
        retained=self.stable.sample(k,env=env)
        self.retained_sampled+=k
        self.normal_sampled+=batch_size-k
        self.mixed_batches+=1
        # SB3 ReplayBufferSamples es una NamedTuple (obs, actions, next_obs, dones, rewards).
        fields=[]
        order=torch.randperm(batch_size,device=base.observations.device)
        for a,b in zip(base,retained):
            # SB3 2.9+ puede incluir `discounts=None` en ReplayBufferSamples.
            if a is None and b is None:
                fields.append(None)
            elif isinstance(a,torch.Tensor) and isinstance(b,torch.Tensor):
                fields.append(torch.cat([a,b],dim=0)[order])
            else:
                raise TypeError(f'Campo de ReplayBufferSamples incompatible: {type(a)} / {type(b)}')
        return ReplayBufferSamples(*fields)

print('Estructura de retencion definida.')

# Prueba sin SB3 training: llenado sintetico local y global.
from gymnasium import spaces
_rb=RetentionReplayBuffer(buffer_size=256,
    observation_space=spaces.Box(low=-10,high=10,shape=(4,),dtype=np.float32),
    action_space=spaces.Box(low=-10,high=10,shape=(1,),dtype=np.float32),
    device='cpu',n_envs=1,stable_capacity=64,fraction=.25)
for i in range(150):
    inside=(i%3==0)
    obs=np.array([[.01,.01,.01,.01] if inside else [1.,1.,2.,2.]],dtype=np.float32)
    nex=obs.copy()
    action=np.zeros((1,1),dtype=np.float32)
    reward=np.array([1.],dtype=np.float32)
    done=np.array([0.],dtype=np.float32)
    _rb.add(obs,nex,action,reward,done,[{}])
assert _rb.total_added==150 and _rb.local_added==50 and _rb.stable.size()==50
assert _rb.strict_added==50
s0=_rb.sample(128)
assert s0.observations.shape==(128,4) and _rb.retained_sampled==0
_rb.retention_active=True
s1=_rb.sample(128)
assert s1.observations.shape==(128,4) and s1.actions.shape==(128,1)
assert s1.rewards.shape==(128,1) and _rb.retained_sampled==32
assert _rb.normal_sampled==224 and _rb.mixed_batches==1
assert (s1.observations[:,0].abs()<.35).sum().item()>=32
print('PRUEBAS OK: reserva local, criterios estrictos, activacion, formato SB3 y lotes de 128.')

In [ ]:
def evaluar(modelo,semilla,tratamiento,steps=6000,guardar_traza=False):
    # Mismo entorno físico de prueba; no hay currículo durante la evaluación.
    env=make_agent_env(tratamiento,steps)
    obs,_=env.reset(seed=int(semilla))
    largo=actual=estable=pasos_cerca=cruces=0
    capturas_1s=0; previo_capturado=False; fraccion_local=0
    cumplen={"x":0,"v":0,"theta":0,"omega":0}
    prev_theta=float(env.unwrapped.state[2])
    rmse_acum=cos_acum=esfuerzo=vel_cruce=0.0
    velocidades_cruce=[]; traza=[];termina_riel=False
    for k in range(steps):
        accion,_=modelo.predict(obs,deterministic=True)
        u=float(np.asarray(accion).reshape(-1)[0])
        obs,_,term,trunc,info=env.step(np.array([u],dtype=np.float32))
        x,v,th,w=map(float,env.unwrapped.state)
        dentro=abs(x)<=.20 and abs(v)<=.10 and abs(th)<=.05 and abs(w)<=.10
        cumplen['x']+=int(abs(x)<=.20)
        cumplen['v']+=int(abs(v)<=.10)
        cumplen['theta']+=int(abs(th)<=.05)
        cumplen['omega']+=int(abs(w)<=.10)
        local=(abs(x)<CFG['x_bono_r2_m'] and abs(v)<CFG['v_bono_r2_m_s'] and abs(th)<CFG['theta_bono_rad'] and abs(w)<CFG['omega_bono_r1_rad_s'])
        fraccion_local+=int(local)
        actual=actual+1 if dentro else 0
        if actual==50: capturas_1s+=1
        largo=max(largo,actual); estable+=int(dentro)
        pasos_cerca+=int(abs(th)<.15)
        rmse_acum+=th*th;cos_acum+=math.cos(th);esfuerzo+=u*u*CFG['dt']
        # Cruce próximo a vertical, detección aproximada por cambio de signo y no por wrap de +/-pi
        if (prev_theta*th<=0 and abs(prev_theta)<.6 and abs(th)<.6):
            cruces+=1; velocidades_cruce.append(abs(w))
        prev_theta=th
        if guardar_traza:
            traza.append(dict(t=(k+1)*CFG['dt'],x=x,v=v,theta=th,omega=w,u=u,
                               estricta=int(dentro),equilibrio_local=int(local),cerca_vertical=int(abs(th)<.15)))
        if term or trunc:
            termina_riel=bool(term); break
    n=k+1;env.close()
    resultado=dict(seed=int(semilla),estabilidad_max_s=largo*CFG['dt'],
      ocupacion_pct=100*estable/n,supervivencia_s=n*CFG['dt'],
      rmse_theta_rad=math.sqrt(rmse_acum/n),cos_theta_medio=cos_acum/n,
      esfuerzo_N2s=esfuerzo,esfuerzo_medio_N2=esfuerzo/(n*CFG['dt']),
      fraccion_cerca_vertical=pasos_cerca/n,fin_por_riel=termina_riel,
      fraccion_equilibrio_local_pct=100*fraccion_local/n, capturas_1s=int(capturas_1s),
      **{f'cumple_{nombre}_pct':100*valor/n for nombre,valor in cumplen.items()},
      cruces_vertical=cruces,
      velocidad_angular_cruce_mediana_rad_s=(float(np.median(velocidades_cruce)) if velocidades_cruce else np.nan))
    return (resultado,pd.DataFrame(traza)) if guardar_traza else resultado

def score(row):
    # Score diagnóstico predefinido; no reemplaza la métrica primaria de estabilidad
    return (row['estabilidad_max_s']/(CFG['episode_steps']*CFG['dt'])
            +0.5*row['ocupacion_pct']/100
            +0.25*(row['cos_theta_medio']+1)/2
            -0.2*float(row['fin_por_riel']))
print('Evaluación preparada: métricas físicas originales y observaciones coherentes con el tratamiento.')

## 7. Entrenamiento de las diez políticas con guardado automático en Drive

**No ejecutes esta sección si ya están las diez ramas completas.** Cada checkpoint se guarda de forma verificable en `MyDrive/N61_VALIDACION_RETENCION_DDPG_R2/seed_<semilla>/<tratamiento>/modelo_<pasos>.zip`.

Si Colab pierde la conexión, vuelve a ejecutar las secciones 1–6 y después esta sección: las ramas **completadas** se reutilizan; las ramas incompletas se reinician porque los estados de optimizador/replay buffer no se preservan para reanudación a mitad de rama.


In [ ]:
def guardar_checkpoint_en_drive(modelo,destino):
    destino=Path(destino);destino.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='n61_ckpt_') as tmp:
        p=Path(tmp)/'modelo.zip';modelo.save(str(p))
        assert p.stat().st_size>1000
        with zipfile.ZipFile(p) as z:assert z.testzip() is None
        q=destino.with_name(destino.name+'.partial');shutil.copy2(p,q)
        with zipfile.ZipFile(q) as z:assert z.testzip() is None
        assert hashlib.sha256(p.read_bytes()).digest()==hashlib.sha256(q.read_bytes()).digest()
        q.replace(destino)
    assert destino.exists() and destino.stat().st_size>1000
    print('    Guardado Drive:',destino.name,flush=True)

class N6Callback(BaseCallback):
    def __init__(self,folder,training_treatment):
        super().__init__()
        self.folder=Path(folder);self.treatment=training_treatment
        self.counters=[];self.pending='equilibrio'
        self.reset_bucket()
    def reset_bucket(self):
        self.bucket=dict(n=0,local=0,cerca=0,estricta=0,v_estricta=0,stages={})
    def _on_step(self):
        info=self.locals.get('infos',[{}])[0]
        b=self.bucket;b['n']+=1
        for dst,src in [('local','local_r2_activado'),('cerca','cerca_vertical'),
                        ('estricta','estricta'),('v_estricta','velocidad_carro_estricta')]:
            b[dst]+=int(info.get(src,0))
        stage=info.get('curriculum_stage','tarea_original')
        b['stages'][stage]=b['stages'].get(stage,0)+1
        t=int(self.model.num_timesteps)
        if t in CFG['checkpoints']:
            replay=self.model.replay_buffer
            b['reserva_local_total']=getattr(replay,'local_added',0)
            b['reserva_estricta_total']=getattr(replay,'strict_added',0)
            b['reserva_size']=getattr(getattr(replay,'stable',None),'size',lambda:0)()
            b['reserva_sampleada_total']=getattr(replay,'retained_sampled',0)
            b['reserva_lotes_mezclados']=getattr(replay,'mixed_batches',0)
            guardar_checkpoint_en_drive(self.model,self.folder/f'modelo_{t}.zip')
            n=b['n'];self.counters.append(dict(checkpoint=t,n_pasos=n,
                r2_local_pct=100*b['local']/n,cerca_vertical_pct=100*b['cerca']/n,
                reserva_local_total=b['reserva_local_total'],reserva_estricta_total=b['reserva_estricta_total'],
                reserva_size=b['reserva_size'],reserva_sampleada_total=b['reserva_sampleada_total'],
                reserva_lotes_mezclados=b['reserva_lotes_mezclados'],
                estricta_pct=100*b['estricta']/n,v_estricta_pct=100*b['v_estricta']/n,
                **{f'etapa_{k}_pasos':b['stages'].get(k,0) for k in ETAPAS}))
            p=self.folder/'auditoria_estados_entrenamiento.csv';q=p.with_name(p.name+'.partial')
            pd.DataFrame(self.counters).to_csv(q,index=False);q.replace(p)
            print(f'  checkpoint {t:,}: cerca={100*b["cerca"]/n:.2f}%, '+
                  f'estabilidad={100*b["estricta"]/n:.2f}%',flush=True)
            self.reset_bucket()
            if self.treatment=='N6_RETENCION' and t==RETENCION['activate_at']:
                self.model.replay_buffer.retention_active=True
                print('  RETENCION ACTIVADA: 25% del lote de reserva local',flush=True)
            if t<CFG['total_steps']:
                stage_next={40000:'recuperacion',80000:'mezcla_swingup',120000:'tarea_original'}[t]
                # DummyVecEnv -> Monitor -> StateAudit -> CurriculumReset.
                solicitar_etapa(self.training_env.envs[0],stage_next,t)
                print('  etapa solicitada:',stage_next,'; activa al siguiente reinicio',flush=True)
        return True

def entrenar(seed,tratamiento):
    folder=OUT/f'seed_{seed}'/tratamiento;folder.mkdir(parents=True,exist_ok=True)
    regpath=folder/'registro.json'
    if regpath.exists():
        reg=json.loads(regpath.read_text())
        if (reg.get('completed') and reg.get('configuracion')==H02
            and reg.get('tipo')==tratamiento and reg.get('steps_end')==CFG['total_steps']
            and reg.get('retencion')==(RETENCION if tratamiento=='N6_RETENCION' else None)
            and all((folder/f'modelo_{cp}.zip').exists() for cp in CFG['checkpoints'])
            and (folder/'auditoria_estados_entrenamiento.csv').exists()
            and all((folder/f'modelo_{cp}.zip').stat().st_size > 1000 for cp in CFG['checkpoints'])
            and all(zipfile.ZipFile(folder/f'modelo_{cp}.zip').testzip() is None for cp in CFG['checkpoints'])):
            print('REUTILIZADO:',seed,tratamiento,flush=True);return
    set_random_seed(int(seed))
    env=Monitor(make_agent_env(tratamiento,CFG['episode_steps'],train_audit=True),
                filename=str(folder/'monitor'))
    env.reset(seed=int(seed))
    noise=NormalActionNoise(mean=np.zeros(1),sigma=np.ones(1)*H02['sigma'])
    replay_cfg=({'replay_buffer_class':RetentionReplayBuffer,
                 'replay_buffer_kwargs':{'stable_capacity':RETENCION['capacity'],
                     'fraction':RETENCION['fraction']}} if tratamiento=='N6_RETENCION' else {})
    model=DDPG('MlpPolicy',env,**replay_cfg,learning_rate=H02['learning_rate'],gamma=H02['gamma'],
        tau=H02['tau'],batch_size=H02['batch_size'],buffer_size=100000,
        learning_starts=CFG['learning_starts'],train_freq=(1,'step'),gradient_steps=1,
        action_noise=noise,policy_kwargs={'net_arch':H02['net_arch'],'n_critics':1},
        seed=int(seed),device='cpu',verbose=0)
    assert model.num_timesteps==0 and model.replay_buffer.size()==0
    print('ENTRENANDO:',seed,tratamiento,flush=True)
    t0=time.perf_counter()
    try:
        model.learn(total_timesteps=CFG['total_steps'],reset_num_timesteps=True,
                    callback=N6Callback(folder,tratamiento),progress_bar=False)
        assert model.num_timesteps==CFG['total_steps']
        assert all((folder/f'modelo_{cp}.zip').exists() for cp in CFG['checkpoints'])
    finally:env.close()
    reg=dict(completed=True,semilla=int(seed),tipo=tratamiento,
             configuracion=H02,steps_start=0,steps_end=CFG['total_steps'],
             learning_starts=CFG['learning_starts'],
             retencion=(RETENCION if tratamiento=='N6_RETENCION' else None),minutes=round((time.perf_counter()-t0)/60,3))
    q=regpath.with_name(regpath.name+'.partial')
    q.write_text(json.dumps(reg,indent=2,ensure_ascii=False),encoding='utf-8');q.replace(regpath)
    print('COMPLETADO:',seed,tratamiento,flush=True)

print('Estado de Drive:')
for seed in CFG['train_seeds']:
    for treatment in TRATAMIENTOS:
        p=OUT/f'seed_{seed}'/treatment
        print(seed,treatment,[cp for cp in CFG['checkpoints'] if (p/f'modelo_{cp}.zip').exists()])
for seed in CFG['train_seeds']:
    for treatment in TRATAMIENTOS:
        entrenar(seed,treatment)
print('DIEZ ENTRENAMIENTOS COMPLETADOS.')


## 8. Validación de checkpoints — condiciones originales (sin currículo)

Si Colab se desconecta **después de entrenar**, ejecuta las celdas de instalación y definiciones, pero omite la sección 7: la validación carga los 40 checkpoints de Drive y no reentrena. Se utilizarán exclusivamente los cinco episodios **8200–8204** para elegir cada uno de los diez modelos.


In [ ]:
esperados=[OUT/f'seed_{seed}'/tratamiento/f'modelo_{cp}.zip'
           for seed in CFG['train_seeds'] for tratamiento in TRATAMIENTOS
           for cp in CFG['checkpoints']]
faltantes=[str(p) for p in esperados if not p.exists()]
print(f'Checkpoints persistentes en Drive: {len(esperados)-len(faltantes)}/{len(esperados)}')
assert not faltantes, 'Faltan checkpoints en Drive:\n'+'\n'.join(faltantes)
rows=[]
for seed in CFG['train_seeds']:
    for etiqueta in TRATAMIENTOS:
        for cp in CFG['checkpoints']:
            path=OUT/f'seed_{seed}'/etiqueta/f'modelo_{cp}.zip'
            assert path.exists(),f'Falta {path}'
            model=DDPG.load(str(path),device='cpu')
            assert model.num_timesteps==cp and model.batch_size==H02['batch_size']
            for evalseed in CFG['val_seeds']:
                res=evaluar(model,evalseed,etiqueta,CFG['episode_steps'])
                rows.append(dict(train_seed=seed,tratamiento=etiqueta,checkpoint=cp,
                                 eval_seed=evalseed,**{k:v for k,v in res.items() if k!='seed'}))
valid=pd.DataFrame(rows)
assert len(valid)==len(CFG['train_seeds'])*len(TRATAMIENTOS)*len(CFG['checkpoints'])*len(CFG['val_seeds'])
valid.to_csv(OUT/'validacion_checkpoints.csv',index=False)
metrics_val=['estabilidad_max_s','ocupacion_pct','fraccion_cerca_vertical',
             'rmse_theta_rad','cumple_v_pct','cumple_x_pct']
summary=(valid.groupby(['train_seed','tratamiento','checkpoint'],as_index=False)[metrics_val].mean())
summary.to_csv(OUT/'resumen_validacion.csv',index=False)
selected=(summary.sort_values(
    ['train_seed','tratamiento','estabilidad_max_s','ocupacion_pct',
     'fraccion_cerca_vertical','rmse_theta_rad','cumple_v_pct','checkpoint'],
    ascending=[True,True,False,False,False,True,False,False],kind='stable')
    .drop_duplicates(['train_seed','tratamiento']).copy())
assert len(selected)==len(CFG['train_seeds'])*len(TRATAMIENTOS)
selected.to_csv(OUT/'checkpoints_seleccionados_SOLO_VALIDACION.csv',index=False)
(OUT/'seleccion_sha256.txt').write_text(hashlib.sha256(
    (OUT/'checkpoints_seleccionados_SOLO_VALIDACION.csv').read_bytes()).hexdigest()+'\n')
print(selected.to_string(index=False))
print('Selección congelada; NO utilizar datos de pruebas para modificarla.')

## 9. Evaluación final reservada — semillas 9980–9989

Ninguna evaluación reservada se utiliza para elegir checkpoints. Unidad inferencial: **cinco semillas de entrenamiento emparejadas**; las diez ejecuciones por política son medidas repetidas. Se informa la comparación por semilla y la evolución de 40K a 160K por separado.


In [ ]:
rows=[]
for row in selected.itertuples(index=False):
    path=OUT/f'seed_{row.train_seed}'/row.tratamiento/f'modelo_{row.checkpoint}.zip'
    model=DDPG.load(str(path),device='cpu')
    for evalseed in CFG['test_seeds']:
        guardar=(evalseed==CFG['test_seeds'][0])
        resultado=evaluar(model,evalseed,row.tratamiento,CFG['episode_steps'],guardar_traza=guardar)
        if guardar:
            vals,traza=resultado
            traza.to_csv(OUT/f'trayectoria_{row.tratamiento}_train_{row.train_seed}_eval_{evalseed}.csv',index=False)
        else:vals=resultado
        rows.append(dict(train_seed=int(row.train_seed),tratamiento=row.tratamiento,
             checkpoint=int(row.checkpoint),eval_seed=evalseed,
             **{k:v for k,v in vals.items() if k!='seed'}))
test=pd.DataFrame(rows)
assert len(test)==len(CFG['train_seeds'])*len(TRATAMIENTOS)*len(CFG['test_seeds']) and set(test.eval_seed)==set(CFG['test_seeds'])
test.to_csv(OUT/'prueba_final_9980_9989.csv',index=False)
metrics=['estabilidad_max_s','ocupacion_pct','rmse_theta_rad','cumple_x_pct',
         'cumple_v_pct','cumple_theta_pct','cumple_omega_pct',
         'supervivencia_s','esfuerzo_medio_N2','fraccion_cerca_vertical']
by_policy=test.groupby(['train_seed','tratamiento'],as_index=False)[metrics].mean()
by_policy.to_csv(OUT/'resumen_por_politica.csv',index=False)
agg=by_policy.groupby('tratamiento')[metrics].agg(['mean','std'])
agg.columns=['_'.join(x) for x in agg.columns]
agg.to_csv(OUT/'resumen_entre_semillas.csv')
pivot=by_policy.pivot(index='train_seed',columns='tratamiento',values='estabilidad_max_s')
assert set(pivot.columns)==set(TRATAMIENTOS)
deltas=(pivot['N6_RETENCION']-pivot['N5_CURRICULO']).rename('delta_RETENCION_menos_CURRICULO_s')
deltas.to_csv(OUT/'diferencias_pareadas_RETENCION_menos_CURRICULO.csv',header=True)
exitos=by_policy.assign(exito_100s=by_policy.estabilidad_max_s>=100.)
tasas=exitos.groupby('tratamiento')['exito_100s'].sum()
reglas=dict(mejora_media_10s=bool(deltas.mean()>=10),
            mejora_4_de_5_semillas=bool((deltas>0).sum()>=4),
            sin_perder_exitos_100s=bool(tasas['N6_RETENCION']>=tasas['N5_CURRICULO']))
decision=('SENAL FAVORABLE DE REPRODUCIBILIDAD N6.1 (n=5); REQUIERE VALIDACION AMPLIADA' if all(reglas.values())
          else 'N6.1: NO SE CONFIRMA MEJORA REPRODUCIBLE SEGUN CRITERIOS')
(OUT/'decision_predefinida.json').write_text(json.dumps(dict(
    reglas=reglas,decision=decision,n_semillas=len(CFG['train_seeds']),
    delta_media_s=float(deltas.mean()),
    mejoras_por_semilla=int((deltas>0).sum()),
    exitos_100s={k:int(v) for k,v in tasas.items()}),
    ensure_ascii=False,indent=2),encoding='utf-8')
print('Resultados por política:\n',by_policy.round(3).to_string(index=False))
print('Diferencias pareadas RETENCION - CURRICULO:\n',deltas.round(3).to_string())
print('Resumen delta: media=',round(float(deltas.mean()),3),'mediana=',round(float(deltas.median()),3),'min=',round(float(deltas.min()),3),'max=',round(float(deltas.max()),3))
print('Éxitos ≥100s:',tasas.to_dict())
print('Reglas:',reglas,'→',decision)
print('ADVERTENCIA: 5 semillas independientes; 50 episodios por tratamiento NO son 50 réplicas.')

## 10. Integridad, evolución y retención del aprendizaje; gráficas y exportación

Se exigen **40 checkpoints**, **200** episodios de validación y **100** episodios finales. La evaluación de checkpoints se contrasta con el último checkpoint para estudiar el deterioro durante el entrenamiento. Los valores `NaN` solo se aceptan en la métrica de cruces verticales cuando no se registró ningún cruce válido. Se genera un ZIP descargable con los modelos, manifiesto, auditorías y resultados.


In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4.5))
for j,m in enumerate(['estabilidad_max_s','ocupacion_pct']):
    for seed in CFG['train_seeds']:
        part=by_policy[by_policy.train_seed==seed].set_index('tratamiento')
        axs[j].plot(list(TRATAMIENTOS),part.loc[list(TRATAMIENTOS),m].values,
                    marker='o',label=f'Semilla {seed}')
    axs[j].set_title('Estabilidad continua máxima' if j==0 else 'Ocupación estricta')
    axs[j].set_ylabel('segundos' if j==0 else '%')
    axs[j].grid(alpha=.25)
axs[0].legend(fontsize=8);fig.tight_layout()
fig.savefig(OUT/'comparacion_N6.png',dpi=150)
plt.show()
for name,df in [('validacion',valid),('final',test),('por_politica',by_policy)]:
    faltantes=df.isna().sum()
    permitidos={'velocidad_angular_cruce_mediana_rad_s'}
    inesperados=[c for c in faltantes[faltantes>0].index if c not in permitidos]
    assert not inesperados,(name,inesperados)
    if 'velocidad_angular_cruce_mediana_rad_s' in df:
        mask=df['velocidad_angular_cruce_mediana_rad_s'].isna()
        assert (df.loc[mask,'cruces_vertical']==0).all()
    print(name,'filas:',len(df),'faltantes:',faltantes[faltantes>0].to_dict())
audit=[]
for seed in CFG['train_seeds']:
    for et in TRATAMIENTOS:
        p=OUT/f'seed_{seed}'/et/'auditoria_estados_entrenamiento.csv'
        assert p.exists(),f'Falta auditoría: {p}'
        a=pd.read_csv(p)
        assert len(a)==len(CFG['checkpoints']) and list(a.checkpoint)==CFG['checkpoints']
        assert a.n_pasos.sum()==CFG['total_steps']
        assert a[['r2_local_pct','cerca_vertical_pct','estricta_pct','v_estricta_pct']].notna().all().all()
        a.insert(0,'tratamiento',et);a.insert(0,'train_seed',seed)
        audit.append(a)
audit=pd.concat(audit,ignore_index=True)
audit.to_csv(OUT/'auditoria_estados_entrenamiento_TODOS.csv',index=False)
assert (audit.loc[audit.tratamiento=='N5_CURRICULO','reserva_sampleada_total']==0).all()
assert (audit.loc[(audit.tratamiento=='N6_RETENCION')&(audit.checkpoint==40000),'reserva_sampleada_total']==0).all()
retencion_final=audit[(audit.tratamiento=='N6_RETENCION')&(audit.checkpoint==160000)]
print('Retención por semilla a 160K:')
print(retencion_final[['train_seed','reserva_local_total','reserva_estricta_total',
    'reserva_size','reserva_sampleada_total','reserva_lotes_mezclados']].to_string(index=False))
assert (retencion_final.reserva_lotes_mezclados>0).all(), 'Reserva sin uso en al menos una semilla; no se puede interpretar tratamiento como aplicado'
assert (retencion_final.reserva_sampleada_total>0).all(), 'No hubo muestras retenidas' 
print('Auditoría durante entrenamiento (último segmento por modelo):')
print(audit[audit.checkpoint==CFG['total_steps']][['train_seed','tratamiento','r2_local_pct',
                                     'cerca_vertical_pct','estricta_pct','v_estricta_pct']]
      .round(4).to_string(index=False))
# Diagnóstico: no confundir política seleccionada con el último checkpoint.
val_by_ckpt=(valid.groupby(['train_seed','tratamiento','checkpoint'],as_index=False)
             ['estabilidad_max_s'].mean())
val_by_ckpt.to_csv(OUT/'evolucion_estabilidad_validacion_por_checkpoint.csv',index=False)
final_ckpt=val_by_ckpt[val_by_ckpt.checkpoint==CFG['total_steps']][
    ['train_seed','tratamiento','estabilidad_max_s']].rename(
    columns={'estabilidad_max_s':'estabilidad_160K_validacion_s'})
retencion_diagnostico=(selected[['train_seed','tratamiento','checkpoint','estabilidad_max_s']]
    .rename(columns={'checkpoint':'checkpoint_seleccionado','estabilidad_max_s':'estabilidad_mejor_validacion_s'})
    .merge(final_ckpt,on=['train_seed','tratamiento'],validate='one_to_one'))
retencion_diagnostico['caida_mejor_a_160K_s']=(
    retencion_diagnostico.estabilidad_mejor_validacion_s-
    retencion_diagnostico.estabilidad_160K_validacion_s)
retencion_diagnostico.to_csv(OUT/'diagnostico_retencion_checkpoint_final.csv',index=False)
print('Adquisición y retención (validación):')
print(retencion_diagnostico.round(3).to_string(index=False))
print('Advertencia: los checkpoints elegidos por validación no equivalen a conservar la habilidad hasta 160K.')

manifest=dict(experimento='N61_validacion_independiente_retencion_DDPG_R2',
    pregunta='¿La reserva de experiencias de equilibrio local de N6 mejora consistentemente la estabilidad de DDPG con currículo en 5 semillas nuevas?',
    configuracion=CFG,hiperparametros=H02,tratamientos=TRATAMIENTOS,etapas_curriculo=ETAPAS,
    representacion='Ambas ramas usan [x,v,theta,omega] y el mismo currículo de N5.',
    retencion=RETENCION,criterio_reserva='ambos estados dentro de region local R2 y transición no terminal',
    activacion='retención al completar 40K pasos; cambios de currículo solo al reset',
    recompensa_R2='R1 + bono local uniforme +3 (histórica; inalterada)',
    validacion=CFG['val_seeds'],evaluacion_final=CFG['test_seeds'],
    seleccion='media estricta > ocupacion > fraccion_cerca_vertical > menor RMSE angular > cumple_v > checkpoint tardio',
    unidad_inferencial='semilla de entrenamiento (n=5); diez episodios por política son observaciones repetidas',
    pasos_totales=1600000,
    limite_interpretacion='Validación independiente con cinco semillas nuevas; la reserva local no equivale a estabilidad estricta; el éxito de checkpoint seleccionado no asegura retención a 160K',
    entorno_sha256=hashlib.sha256(ORIGINAL_ENV.encode('utf-8')).hexdigest(),
    software=dict(stable_baselines3=sb3.__version__,gymnasium=gym.__version__,python=platform.python_version()))
(OUT/'manifiesto.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
archive=Path('/content/experimento_N61_VALIDACION_RETENCION_DDPG_R2_resultados.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=4) as z:
    for p in OUT.rglob('*'):
        if p.is_file():z.write(p,p.as_posix())
print('ZIP:',archive,'MB:',round(archive.stat().st_size/1e6,2))
files.download(str(archive))

## 11. Interpretación, seguridad de datos y próximos pasos

- **Unidad estadística:** cada semilla de entrenamiento (n=5). No confundir 100 episodios finales con 100 replicaciones de entrenamiento.
- **Efecto principal:** media de las cinco diferencias pareadas (N6_RETENCION − N5_CURRICULO) en máxima estabilidad estricta continua de checkpoints seleccionados exclusivamente con validación.
- **Retención real:** revisar `diagnostico_retencion_checkpoint_final.csv`; comparar la estabilidad del checkpoint elegido frente al checkpoint de **160K**. Una mejora basada en checkpoints antiguos no resuelve por sí sola el deterioro posterior.
- **Intervención implementada:** 25 % de cada minibatch elegible de la reserva local R2 a partir de 40K; no equivale a experiencia estrictamente estable.
- **Criterios de confirmación:** promedio ≥10 s, mejoras en ≥4 de 5 semillas y sin pérdida de políticas con ≥100 s. Estos umbrales son criterios exploratorios previamente definidos, no pruebas estadísticas de significación.
- **Persistencia:** el ZIP final contiene checkpoints y auditorías. Una interrupción durante una rama sin `registro.json` completo reinicia esa rama desde cero; las ramas completas se reutilizan desde Drive.
- **Después de finalizar:** envía `experimento_N61_VALIDACION_RETENCION_DDPG_R2_resultados.zip` para contrastar los resultados y decidir si se justifica una confirmación más amplia.
